# Bivariate climate map from TerraClimate monthly data

[Live interactive map](https://www.danielpradilla.info/projects/bivariate-climate-map/) · [Milan Janosov's method](https://milanjanosov.substack.com/p/bivariate-climate-map-temperature)

# 1. Define the annual maps

These maps describe one year's annual mean monthly maximum temperature and accumulated precipitation. They do not show annual mean temperature, a year's hottest day, or a climate normal. The default is 2025 for Europe, Asia, Africa, North America, South America, Oceania, Antarctica and World.

Run every cell in order from the repository root. Open the notebook with `uv run --with-requirements requirements-notebook.txt jupyter lab climate-map.ipynb`. Alternatively, install with `python -m pip install -r requirements-notebook.txt`, then open `jupyter lab climate-map.ipynb`. The notebook downloads two original compressed global NetCDF files containing the twelve monthly fields. It reconstructs annual values from those inputs; it does not read browser annual binaries or existing PNGs. Sources stay in `~/.cache/bivariate-climate-map/notebook`; figures and evidence go to `notebook-output/`.

Allow several GB of free disk for the source cache and several GB of available RAM. Sequential monthly decoding bounds memory, but each global float64 annual field alone uses about 299 MB. Downloads depend on the upstream THREDDS service and can take time. Each region is rendered and released before the next region.

In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import sys
from datetime import datetime, timezone

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import netCDF4
import numpy as np
import pandas as pd
import requests
import shapely
from shapely.geometry import shape
from scipy.ndimage import gaussian_filter
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'build_regions.py').is_file():
    raise RuntimeError('Run this notebook from the repository root.')
sys.path.insert(0, str(ROOT / 'scripts'))
from build_data import (BBOX, NATURAL_EARTH_URL, NODATA,
                        TEMPERATURE_PERCENTILES, PRECIPITATION_PERCENTILES,
                        acquire, digest, encode_fields, european_countries,
                        json_value, source_record)
from build_regions import aggregate, annual_url, country_geometries, grid_for_region

YEAR = 2025
REGION_IDS = ['europe', 'asia', 'africa', 'north-america',
              'south-america', 'oceania', 'antarctica', 'world']
CACHE = Path.home() / '.cache' / 'bivariate-climate-map' / 'notebook'
OUTPUT = ROOT / 'notebook-output'
CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
LIVE_DATA = 'https://www.danielpradilla.info/projects/bivariate-climate-map/data/'
BACKGROUND = '#fffdf9'
plt.rcParams.update({'figure.facecolor': BACKGROUND, 'axes.facecolor': BACKGROUND,
                     'font.family': 'sans-serif', 'font.size': 10, 'font.weight': 'light',
                     'text.color': '#28303d', 'axes.labelcolor': '#28303d'})

# 2. Acquire the source and inspect its metadata

TerraClimate v1.1 provides monthly climate fields on a 1/24-degree global grid. A year in a filename does not establish completeness. Inspect the twelve actual dates, dimensions, units, fill values, scale factor and offset in each downloaded file. CF decoding masks missing encoded values before applying `decoded = packed × scale_factor + add_offset`. Use the attributes from these files, not scales from another distribution such as Google Earth Engine.

The shared acquisition helper keeps a URL and SHA-256 receipt and rejects NetCDF/ZIP error documents. Requests and NetCDF reads here are sequential. Natural Earth supplies public-domain country polygons. Cartopy's bundled features are not used, so plotting does not initiate a second boundary download. Source hashes and global metadata are exported with the figures. v1.1 revises older TerraClimate data; numerical or image identity with an older tutorial is not expected.

In [ ]:
paths = {variable: acquire(annual_url(variable, YEAR),
                           CACHE / f'TerraClimate_{variable}_{YEAR}.nc')
         for variable in ('tmax', 'ppt')}
ne_path = acquire(NATURAL_EARTH_URL, CACHE / 'ne_10m_admin_0_countries.zip')
source_evidence = {}
expected_dates = [f'{YEAR}-{month:02d}-01' for month in range(1, 13)]
for variable, path in paths.items():
    with netCDF4.Dataset(path) as dataset:
        field = dataset.variables[variable]
        time = dataset.variables['time']
        dates = [d.strftime('%Y-%m-%d') for d in netCDF4.num2date(
            time[:], time.units, getattr(time, 'calendar', 'standard'))]
        if dates != expected_dates:
            raise ValueError(f'{variable}: incomplete monthly dates: {dates}')
        source_evidence[variable] = {
            **source_record(annual_url(variable, YEAR), path),
            'dates': dates, 'dimensions': list(field.dimensions), 'shape': list(field.shape),
            'variableAttributes': {key: json_value(field.getncattr(key))
                                   for key in field.ncattrs()},
            'timeAttributes': {key: json_value(time.getncattr(key)) for key in time.ncattrs()},
            'globalAttributes': {key: json_value(dataset.getncattr(key))
                                 for key in dataset.ncattrs()},
        }
        display(pd.DataFrame({'variable': variable, 'month': range(1, 13), 'actual_date': dates}))
        display(pd.DataFrame([{'variable': variable, 'shape': list(field.shape),
                               'units': getattr(field, 'units', None),
                               **{key: json_value(getattr(field, key)) for key in
                                  ('scale_factor', 'add_offset', '_FillValue', 'missing_value')
                                  if hasattr(field, key)}}]))
        print(variable, 'source version:', getattr(dataset, 'version', 'not specified'))

# 3. Calculate annual fields from all twelve months

For cell $i$ and year $y$:

$$T_{i,y}=\frac{1}{12}\sum_{m=1}^{12}T^{\max}_{i,y,m},\qquad P_{i,y}=\sum_{m=1}^{12}P_{i,y,m}.$$

Each temperature month contributes equally; there is no weighting by days. Precipitation is the sum of monthly accumulated totals. A cell remains unavailable if either variable has a missing month. `aggregate` decodes one month at a time using the file's CF attributes, checks the native grid and twelve dates, and returns full-precision annual values. It is the same helper used by the continent asset builder.

In [ ]:
tmax, lon, lat, t_details = aggregate(paths['tmax'], 'tmax', YEAR)
ppt, ppt_lon, ppt_lat, p_details = aggregate(paths['ppt'], 'ppt', YEAR)
if not np.array_equal(lon, ppt_lon) or not np.array_equal(lat, ppt_lat):
    raise ValueError('Temperature and precipitation grids differ.')
source_evidence['tmax']['annualCalculation'] = t_details
source_evidence['ppt']['annualCalculation'] = p_details
del ppt_lon, ppt_lat
display(pd.DataFrame([{'variable': v, 'complete_global_cells': detail['completeYearCellsBeforeRegionMask'],
                       'calculation': detail['aggregation']}
                      for v, detail in [('tmax', t_details), ('ppt', p_details)]]))

# 4. Define the exact geographical selections

Europe preserves the original rectangle: 25°W to 60°E, 34°N to 72°N. Its grid uses every native row and column inside that rectangle, including ocean and non-European cells used by smoothing. `european_countries()` clips Natural Earth's `CONTINENT == Europe` geometry to the rectangle. It includes Iceland and Russia inside the window; Turkey, Cyprus and Caucasus countries are tagged Asia and excluded. Overseas components outside the rectangle are excluded. Cropping Europe to its polygon bounds would change the smoothing extent.

Other continents use `country_geometries()` and `grid_for_region()`: unsimplified country polygons determine cell-centre membership; simplified polygons are display borders only. Asia also includes Russia east of 60°E and its antimeridian components. Overseas country components retain their parent continent tags, so these selections are not physical continent partitions. World includes all country geometries, including Russia, at stride 6 (about 28 km at the equator); continents retain stride 1 (about 4 km). Longitude columns are ordered across a minimal circular interval and can extend beyond 180°. Boundaries are geographic context, not a statement on sovereignty.

Downloaded manifests supply the historical breakpoints and the registry supplies projection settings. If a manifest is absent or a local preview download advertises only a subset, use the canonical public live-site manifest and record its actual URL and hash. A local registry must contain all requested region IDs; a historical manifest must retain the complete 1950 through 2025 baseline, without a download selection. No local app data is changed and no annual raster is downloaded from that site.

In [ ]:
manifest_evidence = {}
def load_manifest(relative, suitable):
    local = ROOT / 'data' / relative
    payload = local.read_bytes() if local.is_file() else None
    document = json.loads(payload) if payload is not None else None
    if document is not None and suitable(document):
        origin = f'data/{relative}'
    else:
        origin = LIVE_DATA + relative
        response = requests.get(origin, timeout=(30, 120))
        response.raise_for_status()
        payload = response.content
        document = json.loads(payload)
        if not suitable(document):
            raise ValueError(f'Canonical manifest does not satisfy the requested scope: {origin}')
    manifest_evidence[relative] = {'origin': origin, 'sha256': hashlib.sha256(payload).hexdigest()}
    return document

def complete_history(document):
    return ('downloadSelection' not in document
            and document.get('years') == list(range(1950, 2026))
            and document.get('baseline', {}).get('complete', True))

registry = load_manifest('regions/index.json', lambda document:
                         set(REGION_IDS) <= {entry['id'] for entry in document.get('regions', [])})
regions = {entry['id']: entry for entry in registry['regions']}
if len(REGION_IDS) != len(set(REGION_IDS)) or set(REGION_IDS) - regions.keys():
    raise ValueError('REGION_IDS must be unique registered region IDs.')
geometry, borders, natural_version = country_geometries(ne_path)
eu_geometry, eu_features, eu_version = european_countries(ne_path)
geometry['Europe'] = eu_geometry
borders['Europe'] = eu_features['features']
natural_evidence = {**source_record(NATURAL_EARTH_URL, ne_path),
                    'version': natural_version, 'europeVersion': eu_version,
                    'license': 'Public domain'}

def region_grid(rid):
    if rid != 'europe':
        return grid_for_region(geometry[regions[rid]['label']], lon, lat,
                               6 if rid == 'world' else 1)
    west, south, east, north = BBOX
    rows = np.flatnonzero((lat >= south) & (lat <= north))
    columns = np.flatnonzero((lon >= west) & (lon <= east))
    if (len(rows), len(columns)) != (912, 2040):
        raise ValueError('Europe must retain the original 912 × 2040 rectangle.')
    mask = shapely.intersects_xy(eu_geometry, lon[columns][None, :], lat[rows][:, None])
    return {'rows': rows, 'columns': columns, 'lon': lon[columns], 'lat': lat[rows],
            'mask': mask, 'indices': np.flatnonzero(mask.ravel()).astype('<u4'), 'stride': 1}

manifests = {rid: load_manifest(regions[rid]['metadataFile'], complete_history) for rid in REGION_IDS}
for rid, manifest in manifests.items():
    if manifest['years'] != list(range(1950, 2026)):
        raise ValueError(f'{rid}: fixed historical manifest must cover 1950 through 2025.')
    if 'baseline' in manifest and not manifest['baseline']['complete']:
        raise ValueError(f'{rid}: historical baseline is incomplete.')
    for key in ('temperatureBreaks', 'precipitationBreaks'):
        if len(manifest[key]) != 5 or not np.all(np.isfinite(manifest[key])):
            raise ValueError(f'{rid}: invalid historical {key}.')

# 5. Check the monthly arithmetic at a real cell

Select an actual complete land cell from the first requested region and read its twelve monthly values directly from the upstream NetCDFs. This independent scalar calculation verifies the annual mean and sum returned by the global helper. The location is a grid-cell centre, not a city observation. The code also checks the packed scalar values against the source CF scale and offset. It chooses a complete cell; it does not fill missing months or substitute a nearby value for a requested location.

In [ ]:
sample_grid = region_grid(REGION_IDS[0])
sample_t = tmax[np.ix_(sample_grid['rows'], sample_grid['columns'])]
sample_p = ppt[np.ix_(sample_grid['rows'], sample_grid['columns'])]
sample_valid = sample_grid['mask'] & np.isfinite(sample_t) & np.isfinite(sample_p)
positions = np.flatnonzero(sample_valid)
if not len(positions):
    raise ValueError('The first selected region has no complete cell for the monthly demonstration.')
sr, sc = np.unravel_index(positions[len(positions) // 2], sample_valid.shape)
global_row, global_col = int(sample_grid['rows'][sr]), int(sample_grid['columns'][sc])
monthly = {}
packed_monthly = {}
for variable, path in paths.items():
    with netCDF4.Dataset(path) as dataset:
        field = dataset.variables[variable]
        field.set_auto_maskandscale(True)
        monthly[variable] = np.asarray(field[:, global_row, global_col].filled(np.nan), dtype=float)
        if not np.isfinite(monthly[variable]).all():
            raise ValueError('Sample cell must have twelve finite months for both variables.')
        scale = float(getattr(field, 'scale_factor', 1))
        offset = float(getattr(field, 'add_offset', 0))
        field.set_auto_maskandscale(False)
        packed_monthly[variable] = np.asarray(field[:, global_row, global_col])
        np.testing.assert_allclose(packed_monthly[variable] * scale + offset,
                                   monthly[variable], rtol=1e-12, atol=1e-10)
point = {'region': REGION_IDS[0], 'globalRow': global_row, 'globalColumn': global_col,
         'longitude': float((lon[global_col] + 180) % 360 - 180),
         'latitude': float(lat[global_row]), 'dates': expected_dates,
         'monthlyDecoded': {v: a.tolist() for v, a in monthly.items()},
         'monthlyPacked': {v: a.tolist() for v, a in packed_monthly.items()},
         'independentTmaxMeanC': float(monthly['tmax'].mean()),
         'independentPptSumMm': float(monthly['ppt'].sum()),
         'aggregateTmaxC': float(tmax[global_row, global_col]),
         'aggregatePptMm': float(ppt[global_row, global_col])}
np.testing.assert_allclose([point['independentTmaxMeanC'], point['independentPptSumMm']],
                           [point['aggregateTmaxC'], point['aggregatePptMm']],
                           rtol=1e-12, atol=1e-9)
display(pd.DataFrame({'date': expected_dates, 'packed_tmax': packed_monthly['tmax'],
                      'decoded_tmax_C': monthly['tmax'], 'packed_ppt': packed_monthly['ppt'],
                      'decoded_ppt_mm': monthly['ppt']}))
display(pd.DataFrame([point]).drop(columns=['dates', 'monthlyDecoded', 'monthlyPacked']))
del sample_grid, sample_t, sample_p, sample_valid, positions

# 6. Encode raw values and compare threshold choices

The app stores raw temperature as signed int16 $\operatorname{rint}(100T)$ and precipitation as $\operatorname{rint}(P)$, with ties rounded to even. Values decode to 0.01°C and 1 mm precision. Missing pairs use -32768. Quantization happens before deriving or applying thresholds, so cells close to a boundary can change class compared with full precision.

Temperature percentile positions are 0, 20, 45, 70 and 100; precipitation positions are 0, 60, 78, 90 and 100. These unequal intervals follow Janosov's method. Annual-only thresholds below are calculated from this year's encoded raw land cells with NumPy's linear quantile interpolation. Historical thresholds are **reused from the published 1950 through 2025 manifests**, not recalculated from 76 years by this notebook. The builders pool encoded raw cell-year observations with integer histograms; a pooled percentile is not the mean of annual percentiles.

Each valid cell-year has equal weight. These percentiles are not area weighted: cells at high latitudes cover less area and still receive the same weight. Each region has its own fixed breaks. Use World for a shared cross-continent scale. The first and last breakpoints describe raw ranges; classification uses only the three internal cutpoints, with equality assigned to the higher class. Values outside the raw range retain the outer class.

A comparison between annual-only and historical classes measures the effect of thresholds on the same smoothed fields. It is not image agreement with a tutorial or a previous TerraClimate release. The code computes and displays the agreements and coverage instead of supplying a prewritten numerical interpretation.

# 7. Smooth continuous fields before classifying

For each region, form the joint valid mask $M$ from polygon membership and complete annual temperature and precipitation. For either continuous field $F$, let $\mu$ be its full-precision mean on $M$. Then:

$$F^{*}=G_{\sigma=2}\bigl(\operatorname{where}(M,F,\mu)\bigr).$$

Use SciPy's Gaussian filter with reflect edges and its default truncation at four sigma, restore the original mask, then quantize with `encode_fields`. Apply breakpoints to these encoded smoothed values. Raw encoded fields supply both annual and historical thresholds; classes and colours are never averaged.

Filling ocean, non-region and incomplete cells with the region's annual mean biases coastal and mask-edge values toward that mean. This is a visual treatment. Raw annual values remain the inspection reference. Sigma 2 means two grid cells: roughly 8 km at the equator for continents and 56 km for stride-6 World, with distances changing by latitude. Europe keeps its entire original rectangle during this operation. Missing cells remain missing after smoothing, including coastal cells. v1.1 has Antarctic data; actual annual coverage is calculated below.

In [ ]:
PALETTE = np.array([
    ['#3a9dbf', '#2a7d82', '#1e6050', '#164427'],
    ['#7ab5c8', '#6d9490', '#5e7a62', '#4a5a30'],
    ['#b0cdd6', '#a8b5aa', '#9e9278', '#9a6e3a'],
    ['#d8d0c8', '#dbbfa4', '#dc9e72', '#dd6a2b'],
])
# class = temperature class + 4 * precipitation class; dry is the bottom legend row.
CLASS_RGBA = np.array([to_rgba(PALETTE[3 - p, t]) for p in range(4) for t in range(4)])

def classify(encoded, valid, t_breaks, p_breaks):
    result = np.full(valid.shape, -1, dtype=np.int8)
    t_class = np.searchsorted(np.asarray(t_breaks)[1:4],
                              encoded[..., 0][valid].astype(float) / 100, side='right')
    p_class = np.searchsorted(np.asarray(p_breaks)[1:4],
                              encoded[..., 1][valid], side='right')
    result[valid] = t_class + 4 * p_class
    return result

def calculate_region(rid, grid):
    t = tmax[np.ix_(grid['rows'], grid['columns'])]
    p = ppt[np.ix_(grid['rows'], grid['columns'])]
    valid = grid['mask'] & np.isfinite(t) & np.isfinite(p)
    if not valid.any():
        raise ValueError(f'{rid}: no complete annual land cells; cannot generate a climate map.')
    raw = encode_fields(t, p, valid)
    annual_t = np.percentile(raw[..., 0][valid], TEMPERATURE_PERCENTILES, method='linear') / 100
    annual_p = np.percentile(raw[..., 1][valid], PRECIPITATION_PERCENTILES, method='linear')
    fixed_t = np.asarray(manifests[rid]['temperatureBreaks'], dtype=float)
    fixed_p = np.asarray(manifests[rid]['precipitationBreaks'], dtype=float)
    means = [float(t[valid].mean()), float(p[valid].mean())]
    smooth_t = gaussian_filter(np.where(valid, t, means[0]), sigma=2, mode='reflect', truncate=4)
    smooth_p = gaussian_filter(np.where(valid, p, means[1]), sigma=2, mode='reflect', truncate=4)
    smooth = encode_fields(smooth_t, smooth_p, valid)
    if not np.array_equal(raw[..., 0] == NODATA, smooth[..., 0] == NODATA):
        raise ValueError(f'{rid}: smoothing changed the valid mask.')
    fixed_classes = classify(smooth, valid, fixed_t, fixed_p)
    annual_classes = classify(smooth, valid, annual_t, annual_p)
    raw_fixed_classes = classify(raw, valid, fixed_t, fixed_p)
    stats = {
        'region': rid, 'year': YEAR, 'stride': grid['stride'],
        'gridShape': list(valid.shape), 'landCells': int(grid['mask'].sum()),
        'validCells': int(valid.sum()), 'incompleteLandCells': int((grid['mask'] & ~valid).sum()),
        'coveragePercent': float(100 * valid.sum() / grid['mask'].sum()),
        'annualTemperatureBreaksC': annual_t.tolist(), 'annualPrecipitationBreaksMm': annual_p.tolist(),
        'historicalTemperatureBreaksC': fixed_t.tolist(), 'historicalPrecipitationBreaksMm': fixed_p.tolist(),
        'annualVersusHistoricalSmoothClassAgreementPercent': float(
            100 * np.mean(annual_classes[valid] == fixed_classes[valid])),
        'rawVersusSmoothHistoricalClassAgreementPercent': float(
            100 * np.mean(raw_fixed_classes[valid] == fixed_classes[valid])),
        'historicalClassCounts': np.bincount(fixed_classes[valid], minlength=16).tolist(),
        'rawTmaxRangeC': [float(raw[..., 0][valid].min() / 100), float(raw[..., 0][valid].max() / 100)],
        'rawPptRangeMm': [int(raw[..., 1][valid].min()), int(raw[..., 1][valid].max())],
        'meanFillTmaxC': means[0], 'meanFillPptMm': means[1],
        'rawToSmoothMeanAbsoluteTmaxChangeC': float(np.mean(np.abs(smooth_t[valid] - t[valid]))),
        'rawToSmoothMeanAbsolutePptChangeMm': float(np.mean(np.abs(smooth_p[valid] - p[valid]))),
        'thresholdReference': manifests[rid]['thresholdReference'],
        'pooledHistoricalCellYears': manifests[rid]['pooledValidCellYears'],
        'projection': regions[rid]['projection'],
        'gridWestUnwrapped': float(grid['lon'][0]), 'gridEastUnwrapped': float(grid['lon'][-1]),
        'gridNorth': float(grid['lat'][0]), 'gridSouth': float(grid['lat'][-1]),
        'scope': manifests[rid].get('scope'),
    }
    return fixed_classes, stats

# 8. Render and export the eight categorical maps

The registry's D3 rotation signs convert to Cartopy centres by negating longitude and latitude: continents use Lambert azimuthal equal-area, Antarctica uses south-polar stereographic, and World uses Equal Earth. Cartopy and D3 implementations may differ in projection parameters and viewport fitting; these PNGs reproduce the fields and categories, not browser pixels.

Use a bounded projected pixel grid and nearest source-cell lookup. This avoids constructing millions of native-cell polygons. To handle Asia and Oceania across the dateline, the source CRS is Plate Carrée centred at the midpoint of each unwrapped longitude interval, with longitude expressed relative to that centre. Inverse projection selects one native cell's categorical colour for each display pixel. No interpolation between classes, no artificial dateline column, and no smoothing of colours is applied. Output resolution affects which cells are visible, not their computed categories or statistics. Borders come from the acquired Natural Earth geometry; no Cartopy feature download is requested.

Each PNG uses the fixed historical thresholds and contains its 4×4 legend. Horizontal legend classes increase in annual mean monthly maximum temperature; vertical classes increase in annual precipitation. The printed intervals are the actual regional breakpoints. The first and last classes also include any smoothed values beyond those raw endpoints.

In [ ]:
def projection_for(entry):
    settings = entry['projection']
    central_lon, central_lat = [-float(v) for v in settings['rotate'][:2]]
    kind = settings['type']
    if kind == 'azimuthalEqualArea':
        return ccrs.LambertAzimuthalEqualArea(central_longitude=central_lon, central_latitude=central_lat)
    if kind == 'stereographic':
        return ccrs.Stereographic(central_longitude=central_lon, central_latitude=central_lat)
    if kind == 'equalEarth':
        return ccrs.EqualEarth(central_longitude=central_lon)
    raise ValueError(f'Unsupported registry projection: {kind}')


def projected_bounds(rid, projection, display_geometries):
    if rid == 'world':
        return (*projection.x_limits, *projection.y_limits)
    # Quiet borders use the same geometries for fitting and drawing.
    coords = shapely.get_coordinates(shapely.GeometryCollection(display_geometries))
    projected = projection.transform_points(ccrs.PlateCarree(), coords[:, 0], coords[:, 1])
    finite = np.isfinite(projected[:, :2]).all(axis=1)
    xy = projected[finite, :2]
    if not len(xy):
        raise ValueError(f'{rid}: projection has no finite boundary coordinates.')
    low, high = xy.min(axis=0), xy.max(axis=0)
    pad = (high - low) * 0.025
    return (float(low[0] - pad[0]), float(high[0] + pad[0]),
            float(low[1] - pad[1]), float(high[1] + pad[1]))


def projected_rgba(classes, grid, projection, extent, target_width=1400):
    xmin, xmax, ymin, ymax = extent
    target_height = max(2, int(round(target_width * (ymax - ymin) / (xmax - xmin))))
    x = xmin + (np.arange(target_width) + 0.5) * (xmax - xmin) / target_width
    y = ymin + (np.arange(target_height) + 0.5) * (ymax - ymin) / target_height
    center = float((grid['lon'][0] + grid['lon'][-1]) / 2)
    source_crs = ccrs.PlateCarree(central_longitude=(center + 180) % 360 - 180)
    relative_west = float(grid['lon'][0] - center)
    step = grid['stride'] / 24
    rgba = np.zeros((target_height, target_width, 4), dtype=np.float32)
    # Chunk inverse transforms so plotting does not allocate a native-resolution mesh.
    for start in range(0, target_height, 128):
        xx, yy = np.meshgrid(x, y[start:start + 128])
        source = source_crs.transform_points(projection, xx, yy)
        relative_lon, latitude = source[..., 0], source[..., 1]
        finite = np.isfinite(relative_lon) & np.isfinite(latitude)
        safe_lon = np.where(finite, relative_lon, relative_west)
        safe_lat = np.where(finite, latitude, grid['lat'][0])
        col_fraction = (safe_lon - relative_west) / step
        row_fraction = (grid['lat'][0] - safe_lat) / step
        row = np.floor(row_fraction + 0.5).astype(np.int64)
        col = np.floor(col_fraction + 0.5).astype(np.int64)
        inside = finite & (row_fraction >= -0.5) & (row_fraction <= classes.shape[0] - 0.5)
        if classes.shape[1] * step >= 359.99:
            col %= classes.shape[1]
        else:
            inside &= (col_fraction >= -0.5) & (col_fraction <= classes.shape[1] - 0.5)
        row = np.clip(row, 0, classes.shape[0] - 1)
        col = np.clip(col, 0, classes.shape[1] - 1)
        selected = classes[row, col]
        inside &= selected >= 0
        block = rgba[start:start + len(yy)]
        block[inside] = CLASS_RGBA[selected[inside]]
    return rgba, {'width': target_width, 'height': target_height,
                  'sourcePlateCarreeCenter': center,
                  'lookup': 'inverse projected pixel centre to nearest native categorical cell; no colour interpolation'}


def legend_interval(breaks, index, unit):
    lo, hi = breaks[index:index + 2]
    return f'{lo:g} to {hi:g} {unit}'


def render_region(rid, grid, classes, stats):
    entry = regions[rid]
    projection = projection_for(entry)
    display_geometries = [shape(feature['geometry']) for feature in borders[entry['label']]]
    extent = projected_bounds(rid, projection, display_geometries)
    rgba, render_details = projected_rgba(classes, grid, projection, extent)
    wide = (extent[1] - extent[0]) / (extent[3] - extent[2]) > 1.6
    fig = plt.figure(figsize=(13, 7.2 if wide else 9), facecolor=BACKGROUND)
    ax = fig.add_axes([0.025, 0.17, 0.72, 0.70] if wide else [0.025, 0.15, 0.72, 0.75],
                     projection=projection)
    ax.set_xlim(extent[:2]); ax.set_ylim(extent[2:])
    ax.imshow(rgba, origin='lower', extent=extent, transform=projection,
              interpolation='nearest', zorder=1)
    ax.add_geometries(display_geometries, crs=ccrs.PlateCarree(), facecolor='none',
                      edgecolor='#687177', linewidth=0.35, alpha=0.65, zorder=2)
    ax.spines['geo'].set_visible(False)
    fig.text(0.035, 0.955, f"{entry['label']} · {YEAR}", fontsize=18, weight='light')
    fig.text(0.035, 0.917, 'Annual mean monthly maximum temperature × annual precipitation',
             fontsize=11, color='#666666')
    leg = fig.add_axes([0.805, 0.54, 0.16, 0.23])
    leg.imshow(CLASS_RGBA.reshape(4, 4, 4), origin='lower', interpolation='nearest',
               extent=(0, 4, 0, 4))
    leg.set_xticks(np.arange(1, 4), minor=True)
    leg.set_yticks(np.arange(1, 4), minor=True)
    leg.grid(which='minor', color=BACKGROUND, linewidth=0.6)
    leg.tick_params(which='minor', length=0)
    leg.set_xticks(np.arange(4) + 0.5, ['1', '2', '3', '4'])
    leg.set_yticks(np.arange(4) + 0.5, ['1', '2', '3', '4'])
    leg.set_xlabel('Warmer →', fontsize=8)
    leg.set_ylabel('Wetter ↑', fontsize=8)
    leg.tick_params(length=0, labelsize=8, colors='#28303d')
    for spine in leg.spines.values():
        spine.set_color('#aaaaaa'); spine.set_linewidth(0.5)
    t_breaks, p_breaks = stats['historicalTemperatureBreaksC'], stats['historicalPrecipitationBreaksMm']
    lines = ['Fixed 1950–2025 breaks', '', 'Temperature (horizontal)']
    lines += [f'{i + 1}: {legend_interval(t_breaks, i, "°C")}' for i in range(4)]
    lines += ['', 'Precipitation (vertical)']
    lines += [f'{i + 1}: {legend_interval(p_breaks, i, "mm")}' for i in range(4)]
    fig.text(0.795, 0.445, '\n'.join(lines), fontsize=8.5, va='top', linespacing=1.45)
    fig.text(0.035, 0.115,
             f"{stats['validCells']:,} complete / {stats['landCells']:,} land cells · "
             f"{stats['coveragePercent']:.2f}% coverage · stride {grid['stride']} · Gaussian σ=2 cells",
             fontsize=9, color='#6d6d6d', va='top')
    fig.text(0.035, 0.088,
             'Data: TerraClimate v1.1 (CC0) · Borders: Natural Earth (public domain)\n'
             'Method/palette: Milan Janosov · Earlier work: Muhammad Mohsin Raza; Fil / Philippe Rivière',
             fontsize=8, color='#6d6d6d', linespacing=1.45, va='top')
    fig.text(0.035, 0.040, 'Sources and calculations: climate-map.ipynb · One year, not a climate normal.',
             fontsize=8, color='#6d6d6d', va='top')
    destination = OUTPUT / f'{rid}-climate-{YEAR}.png'
    fig.savefig(destination, dpi=160, facecolor=BACKGROUND,
                metadata={'Title': f'{entry["label"]} annual climate {YEAR}',
                          'Description': 'Genuine monthly TerraClimate inputs; fixed published 1950-2025 thresholds.',
                          'Source': 'https://www.climatologylab.org/terraclimate.html'})
    display(fig)
    plt.close(fig)
    return {'file': str(destination.relative_to(ROOT)), 'sha256': digest(destination),
            'bytes': destination.stat().st_size, 'render': render_details}

In [ ]:
results = []
for rid in REGION_IDS:
    grid = region_grid(rid)
    classes, stats = calculate_region(rid, grid)
    stats['figure'] = render_region(rid, grid, classes, stats)
    results.append(stats)
    display(pd.DataFrame([{
        'region': rid, 'complete_land_cells': stats['validCells'],
        'incomplete_land_cells': stats['incompleteLandCells'],
        'coverage_percent': stats['coveragePercent'],
        'annual_vs_historical_smooth_agreement_percent': stats['annualVersusHistoricalSmoothClassAgreementPercent'],
        'raw_vs_smooth_historical_agreement_percent': stats['rawVersusSmoothHistoricalClassAgreementPercent'],
    }]))
    display(pd.DataFrame({'temperature_percentile': TEMPERATURE_PERCENTILES,
                          'annual_tmax_C': stats['annualTemperatureBreaksC'],
                          'historical_tmax_C': stats['historicalTemperatureBreaksC'],
                          'precipitation_percentile': PRECIPITATION_PERCENTILES,
                          'annual_ppt_mm': stats['annualPrecipitationBreaksMm'],
                          'historical_ppt_mm': stats['historicalPrecipitationBreaksMm']}))
    del grid, classes, stats
    gc.collect()

def serializable(value):
    if isinstance(value, dict):
        return {str(key): serializable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [serializable(item) for item in value]
    return json_value(value)

provenance = {
    'createdAt': datetime.now(timezone.utc).isoformat(), 'year': YEAR, 'regionIds': REGION_IDS,
    'sources': source_evidence, 'naturalEarth': natural_evidence,
    'manifests': manifest_evidence, 'independentMonthlyPoint': point,
    'baseline': {'startYear': 1950, 'endYear': 2025,
                 'mode': 'reuse published manifest quantiles; only the selected year is recomputed here',
                 'weighting': 'unweighted encoded raw land-cell/year observations; NumPy linear quantiles'},
    'smoothing': {'sigmaCells': 2, 'edgeMode': 'reflect', 'truncate': 4,
                  'fill': 'regional annual full-precision joint-valid-field mean; restore joint valid mask'},
    'quantization': {'temperatureScale': 100, 'precipitationScale': 1, 'nodata': NODATA,
                     'rounding': 'numpy.rint; nearest integer, ties to even'},
    'codeSha256': {name: digest(ROOT / name) for name in
                  ['climate-map.ipynb', 'scripts/build_data.py', 'scripts/build_regions.py',
                   'requirements-data.txt', 'requirements-notebook.txt']},
    'dependencies': {name: importlib.metadata.version(name) for name in
                     ['numpy', 'scipy', 'netCDF4', 'shapely', 'matplotlib', 'cartopy', 'pandas']},
    'regions': results,
}
provenance_path = OUTPUT / f'provenance-{YEAR}.json'
provenance_path.write_text(json.dumps(serializable(provenance), indent=2, ensure_ascii=False,
                                     allow_nan=False) + '\n')
summary = pd.DataFrame([{key: item[key] for key in
                        ['region', 'validCells', 'landCells', 'coveragePercent',
                         'annualVersusHistoricalSmoothClassAgreementPercent',
                         'rawVersusSmoothHistoricalClassAgreementPercent']} for item in results])
display(summary)
print(f'Created {len(results)} PNGs and {provenance_path.relative_to(ROOT)}')

# 9. Build the browser assets and the 1950 through 2025 animation

This notebook reconstructs one selected year from monthly input. Building the full animation is a separate acquisition: run `uv run --with-requirements requirements-data.txt python scripts/build_data.py` for the original Europe rectangle, then `uv run --with-requirements requirements-data.txt python scripts/build_data.py --regions` for the other continents and World. The regional command preserves existing European assets. These commands download and process every source year; they are not run by this notebook.

For each year, the builders verify twelve dates, decode monthly CF packing, compute the annual mean/sum, apply the region's unsimplified mask, and export raw and sigma-2 smoothed grids. The continental and World files contain only land positions, using a shared sorted little-endian uint32 `cells.bin.gz` index. Europe retains its dense rectangle. Annual values are interleaved little-endian signed int16 temperature ×100 / precipitation pairs, gzip compressed; missing pairs remain -32768. Source receipts, hashes, monthly dates and CF attributes accompany the exports.

Integer histograms accumulate encoded raw land-cell observations across all 76 years. The builders derive exact NumPy-linear pooled percentiles and publish fixed temperature/precipitation breaks in the manifests. Each region has 76 frames, giving 608 region-year frames across eight views. Complete data must be acquired before a full baseline is claimed. A partial download must advertise only its acquired years and frames while retaining the intended historical baseline definition.

`app.js` fetches the chosen frame on demand, decompresses it, uses the fixed regional thresholds, and displays the selected year's raw or smoothed fields. The animation changes years while keeping thresholds fixed, so physical values can move between classes. Per-year quantiles would conceal part of that change by redefining the classes every year. Masks can vary by year; missing coastal or incomplete cells remain unavailable.

The agreement statistics calculated here describe classification sensitivity. They do not establish a climate trend or causation. Annual weather variation, grid resolution, unweighted percentiles, revised source data and the mean-filled coastal smoothing all limit interpretation. Compare raw values before using the coloured map for a local conclusion.

# 10. Sources, method and licenses

- [Milan Janosov: bivariate climate map](https://milanjanosov.substack.com/p/bivariate-climate-map-temperature), with the original 4×4 palette, unequal percentile positions and sigma-2 smoothing; [video walkthrough](https://www.youtube.com/watch?v=iWRp_dciJsM).
- [Muhammad Mohsin Raza / Datawim: Creating professional bivariate maps in R](https://www.datawim.com/post/creating-professional-bivariate-maps-in-r/).
- [Fil / Philippe Rivière: Pangea bivariate map example](https://fil.github.io/pangea/plot/bivariate-map), with [source code](https://github.com/Fil/pangea).
- [TerraClimate provider and documentation](https://www.climatologylab.org/terraclimate.html): Abatzoglou, Hegewisch, Yin and Kalashinkov (2026), *TerraClimate v1.1, An Updated Global Climate and Climatic Water Balance Dataset with Climate Scenarios*, Scientific Data. The source files record their version, references and CC0 license. See also the [foundational 2018 paper](https://doi.org/10.1038/sdata.2017.191). v1.1 is revised; tutorials using earlier releases can have different values and coverage.
- [Natural Earth 1:10m Admin 0 Countries](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/): [public domain](https://www.naturalearthdata.com/about/terms-of-use/). The downloaded source version is recorded in provenance. Country-continent tags and de facto boundaries define the selections used here.
- [D3, Mike Bostock and contributors](https://d3js.org/): the interactive app uses D3 under the [ISC license](vendor/d3.LICENSE). The notebook uses Cartopy and Matplotlib for independent rendering.

The [live project](https://www.danielpradilla.info/projects/bivariate-climate-map/) serves the manifests used for the historical baseline fallback. PNGs and JSON evidence remain local in `notebook-output/`; source notebook cells are committed without execution outputs.